# WarpFactory — Spacetime Metrics & Warp Drive Physics

A Python port of the MATLAB **WarpFactory** toolkit for computing and analysing  
spacetime metrics from General Relativity — including the **Alcubierre warp drive**,  
**Schwarzschild black hole**, and **Van Den Broeck** metric.

---

## What this notebook does

| Step | What happens |
|------|--------------|
| 1 | Install / detect GPU (CuPy) — falls back to NumPy on CPU |
| 2 | Define all physics code inline (with bug-fixes applied) |
| 3 | Generate a spacetime metric (Alcubierre / Schwarzschild / Van Den Broeck) |
| 4 | Solve the Einstein field equations → stress-energy tensor T^{μν} |
| 5 | Transform T to the locally-flat Eulerian frame |
| 6 | Evaluate Null and Weak energy conditions |
| 7 | Visualise energy density and condition-violation maps |

---

## How to use in Google Colab

1. Open **Runtime → Change runtime type → Hardware accelerator → GPU** (T4 is fine).
2. Click **Runtime → Run all** (or press `Ctrl+F9`).
3. The notebook auto-installs CuPy and uses your GPU when one is detected.
4. To run a specific metric example, scroll to the **Demos** section and run that cell.

### Units used in this notebook

All examples use **code units** where `C = 1` and `G = 1`.  
This keeps coordinates and velocities dimensionless and avoids numerical overflow  
from SI values (C ≈ 3 × 10⁸ m/s).  
To use SI units, change `C = 1.0` → `C = 2.99792458e8` and `G = 1.0` → `G = 6.67430e-11`
in the constants cell, and rescale your grid accordingly.

---
## Cell 1 — Install packages & detect GPU

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=False)

_pip('numpy', 'scipy', 'matplotlib')

# ── GPU detection & CuPy setup ────────────────────────────────────────────────
GPU_AVAILABLE = False
xp = None  # will be set below

try:
    import cupy as _cp
    xp = _cp
    GPU_AVAILABLE = True
    print(f'GPU detected — CuPy already installed.')
    try:
        dev = _cp.cuda.runtime.getDeviceProperties(0)
        print(f"  Device : {dev['name'].decode()}")
        print(f"  Memory : {dev['totalGlobalMem'] / 1e9:.1f} GB")
    except Exception:
        pass
except ImportError:
    print('CuPy not found — trying to install cupy-cuda12x (standard Colab GPU)...')
    _pip('cupy-cuda12x')
    try:
        import cupy as _cp
        xp = _cp
        GPU_AVAILABLE = True
        print('CuPy installed successfully — GPU acceleration ENABLED.')
    except Exception as e:
        import numpy as _np
        xp = _np
        print(f'CuPy unavailable ({e}).  Falling back to NumPy (CPU).')
        print('  To enable GPU: Runtime > Change runtime type > GPU, then re-run.')

import numpy as np          # always available; used for visualisation / I/O
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 120})

def to_np(arr):
    """Move a CuPy / NumPy array to a NumPy array (for plotting)."""
    if GPU_AVAILABLE and hasattr(arr, 'get'):
        return arr.get()
    return np.asarray(arr)

print(f'\nGPU acceleration: {"ENABLED (CuPy)" if GPU_AVAILABLE else "DISABLED (NumPy CPU)"}')

---
## Cell 2 — Core classes & constants

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Any

# ── Physical constants (code units: C = G = 1) ───────────────────────────────
# Change these to SI values if your grid coordinates are in meters/seconds.
C = 1.0          # speed of light  (SI: 2.99792458e8 m/s)
G = 1.0          # gravitational constant  (SI: 6.67430e-11 m³/kg/s²)

@dataclass
class Metric:
    """
    Container for a spacetime metric tensor (or any rank-2 tensor).

    Attributes
    ----------
    tensor   : array shape (4, 4, Nt, Nx, Ny, Nz) — the tensor components.
    coords   : dict with keys 't', 'x', 'y', 'z' — physical coordinate grids.
    scaling  : array (dt, dx, dy, dz) — grid spacings.
    name     : human-readable label.
    index    : 'covariant' (lower indices) or 'contravariant' (upper indices).
    params   : extra parameters (velocity, radius, …)
    """
    tensor  : object  # np/cp ndarray
    coords  : Dict[str, object]
    scaling : object
    name    : str = 'Generic'
    index   : str = 'covariant'
    params  : Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        if self.tensor.shape[:2] != (4, 4):
            raise ValueError(f'Tensor must be (4,4,...), got {self.tensor.shape}')
        sp = self.tensor.shape[2:]
        for k, v in self.coords.items():
            if v.shape != sp:
                raise ValueError(f'Coord {k}: shape {v.shape} != tensor shape {sp}')

print('Core classes loaded.')

---
## Cell 3 — Grid utilities & shape functions

In [ ]:
def create_grid(grid_size, grid_scale):
    """Create a 4-D coordinate meshgrid (t, x, y, z)."""
    Nt, Nx, Ny, Nz = grid_size
    dt, dx, dy, dz = grid_scale
    t, x, y, z = xp.meshgrid(
        xp.arange(Nt, dtype=float) * dt,
        xp.arange(Nx, dtype=float) * dx,
        xp.arange(Ny, dtype=float) * dy,
        xp.arange(Nz, dtype=float) * dz,
        indexing='ij'
    )
    return {'t': t, 'x': x, 'y': y, 'z': z}


def shape_function_alcubierre(r, R, sigma):
    """
    Alcubierre shape function  f(r; R, sigma).
    f ≈ 1 inside radius R, smoothly drops to 0 outside.
    sigma controls the shell thickness.
    """
    num = xp.tanh(sigma * (R + r)) + xp.tanh(sigma * (R - r))
    den = 2.0 * xp.tanh(sigma * R)
    return num / den


def get_minkowski_metric(grid_size):
    """Return the flat Minkowski metric  eta_{μν} = diag(-1, +1, +1, +1)."""
    g = xp.zeros((4, 4) + tuple(grid_size))
    g[0, 0] = -xp.ones(grid_size)
    g[1, 1] =  xp.ones(grid_size)
    g[2, 2] =  xp.ones(grid_size)
    g[3, 3] =  xp.ones(grid_size)
    return g


print('Grid utilities loaded.')

---
## Cell 4 — Metric generators

In [ ]:
def create_alcubierre_metric(grid_size, grid_scale, world_center, v, R, sigma):
    """
    Alcubierre warp drive metric.

    The warp bubble travels in the +x direction at speed v (in units of C).
    R is the bubble radius and sigma controls shell thickness.

    ADM decomposition:
      alpha = 1  (lapse)
      beta^x = -v * f(r_s)  (shift vector)
      gamma_{ij} = delta_{ij}  (flat spatial metric)

    Parameters
    ----------
    grid_size   : (Nt, Nx, Ny, Nz) — Nt must be >= 5 for finite differences.
    grid_scale  : (dt, dx, dy, dz) in code units.
    world_center: (tc, xc, yc, zc) physical centre of the grid.
    v           : bubble velocity in units of C (e.g. 0.5 = half light-speed).
    R           : bubble radius (same units as grid).
    sigma       : shell steepness (larger = thinner shell).
    """
    grid = create_grid(grid_size, grid_scale)
    t, x, y, z = grid['t'], grid['x'], grid['y'], grid['z']

    # Shift by +1 grid spacing to match 1-based MATLAB indexing convention
    t_phys = (t + grid_scale[0]) - world_center[0]
    x_phys = (x + grid_scale[1]) - world_center[1]
    y_phys = (y + grid_scale[2]) - world_center[2]
    z_phys = (z + grid_scale[3]) - world_center[3]

    # Bubble centre position (travels at v*C along x)
    xs = t_phys * v * C

    r  = xp.sqrt((x_phys - xs)**2 + y_phys**2 + z_phys**2)
    fs = shape_function_alcubierre(r, R, sigma)

    beta_x = -v * fs

    g = xp.zeros((4, 4) + t.shape)
    g[0, 0] = -1.0 + beta_x**2
    g[0, 1] = beta_x
    g[1, 0] = beta_x
    g[1, 1] = 1.0
    g[2, 2] = 1.0
    g[3, 3] = 1.0

    return Metric(
        tensor=g,
        coords={'t': t_phys, 'x': x_phys, 'y': y_phys, 'z': z_phys},
        scaling=xp.array(grid_scale, dtype=float),
        name='Alcubierre',
        index='covariant',
        params={'v': v, 'R': R, 'sigma': sigma},
    )


def create_schwarzschild_metric(grid_size, grid_scale, world_center, rs):
    """
    Schwarzschild metric in quasi-Cartesian (isotropic-like) coordinates.

    g_{tt} = -(1 - rs/r),  with cross terms for Cartesian embedding.

    Parameters
    ----------
    rs : Schwarzschild radius  rs = 2 G M / C^2.
         With code units (G=C=1): rs = 2 M.
    """
    if grid_size[0] > 1:
        raise ValueError('Schwarzschild is static — use Nt=1 (or Nt up to 1).')

    grid = create_grid(grid_size, grid_scale)
    t, x, y, z = grid['t'], grid['x'], grid['y'], grid['z']

    x_phys = (x + grid_scale[1]) - world_center[1]
    y_phys = (y + grid_scale[2]) - world_center[2]
    z_phys = (z + grid_scale[3]) - world_center[3]

    epsilon = 1e-10
    r  = xp.sqrt(x_phys**2 + y_phys**2 + z_phys**2) + epsilon

    g = get_minkowski_metric(grid_size)

    factor     = 1.0 - rs / r
    inv_factor = 1.0 / (factor + epsilon)
    r2  = r**2
    x2, y2, z2 = x_phys**2, y_phys**2, z_phys**2

    g[0, 0] = -factor
    g[1, 1] = (x2 * inv_factor + y2 + z2) / r2
    g[2, 2] = (x2 + y2 * inv_factor + z2) / r2
    g[3, 3] = (x2 + y2 + z2 * inv_factor) / r2

    cross = rs / (r**3 * (factor + epsilon))
    g[1, 2] = g[2, 1] = cross * x_phys * y_phys
    g[1, 3] = g[3, 1] = cross * x_phys * z_phys
    g[2, 3] = g[3, 2] = cross * y_phys * z_phys

    return Metric(
        tensor=g,
        coords={'t': t, 'x': x_phys, 'y': y_phys, 'z': z_phys},
        scaling=xp.array(grid_scale, dtype=float),
        name='Schwarzschild',
        index='covariant',
        params={'rs': rs},
    )


def create_van_den_broeck_metric(
    grid_size, grid_scale, world_center,
    v, R1, sigma1, R2, sigma2, A
):
    """
    Van Den Broeck warp drive metric.

    Reduces the exotic energy requirement by expanding space inside the bubble.

    Parameters
    ----------
    v           : bubble velocity (units of C).
    R1, sigma1  : expansion region radius and steepness.
    R2, sigma2  : shift-vector region radius and steepness.
    A           : spatial expansion factor.
    """
    grid = create_grid(grid_size, grid_scale)
    t, x, y, z = grid['t'], grid['x'], grid['y'], grid['z']

    t_phys = (t + grid_scale[0]) - world_center[0]
    x_phys = (x + grid_scale[1]) - world_center[1]
    y_phys = (y + grid_scale[2]) - world_center[2]
    z_phys = (z + grid_scale[3]) - world_center[3]

    v_eff = v * (1 + A)**2
    xs    = t_phys * v_eff * C
    r     = xp.sqrt((x_phys - xs)**2 + y_phys**2 + z_phys**2)

    B       = 1.0 + shape_function_alcubierre(r, R1, sigma1) * A
    fs_val  = shape_function_alcubierre(r, R2, sigma2) * v

    B_sq       = B**2
    shift_term = -B_sq * fs_val

    g = xp.zeros((4, 4) + t.shape)
    g[0, 0] = -(1.0 - B_sq * fs_val**2)
    g[0, 1] = shift_term
    g[1, 0] = shift_term
    g[1, 1] = B_sq
    g[2, 2] = B_sq
    g[3, 3] = B_sq

    return Metric(
        tensor=g,
        coords={'t': t_phys, 'x': x_phys, 'y': y_phys, 'z': z_phys},
        scaling=xp.array(grid_scale, dtype=float),
        name='Van Den Broeck',
        index='covariant',
        params={'v': v, 'R1': R1, 'sigma1': sigma1, 'R2': R2, 'sigma2': sigma2, 'A': A},
    )


print('Metric generators loaded.')

---
## Cell 5 — Finite-difference derivatives (4th-order central)

In [ ]:
def derive_1st_4th_order(A, axis, delta):
    """
    First partial derivative of array A along 'axis' using a 4th-order central
    finite-difference stencil:

        dA/dx ≈ (-A_{+2} + 8A_{+1} - 8A_{-1} + A_{-2}) / (12 dx)

    Boundary points (0, 1, -2, -1) are filled with the nearest valid interior
    value (index 2 or -3) — first-order accurate at edges, but prevents crashes.

    The axis must have at least 5 points; smaller dimensions return zero.
    """
    ndim = A.ndim
    if A.shape[axis] < 5:
        return xp.zeros_like(A)

    vp2 = xp.roll(A, -2, axis=axis)
    vm2 = xp.roll(A,  2, axis=axis)
    vp1 = xp.roll(A, -1, axis=axis)
    vm1 = xp.roll(A,  1, axis=axis)

    B = (-(vp2 - vm2) + 8.0 * (vp1 - vm1)) / (12.0 * delta)

    def idx(i):
        sl = [slice(None)] * ndim
        sl[axis] = i
        return tuple(sl)

    lv = B[idx(2)].copy()
    rv = B[idx(-3)].copy()
    B[idx(0)] = lv;  B[idx(1)] = lv
    B[idx(-1)] = rv; B[idx(-2)] = rv
    return B


print('Finite-difference solver loaded.')

---
## Cell 6 — Tensor calculations  
(Christoffel → Ricci → Einstein → stress-energy)

> **GPU note**: these are the computationally heavy routines.  
> With CuPy enabled, the element-wise operations on large arrays run on the GPU.

In [ ]:
# ── Metric inverse ────────────────────────────────────────────────────────────

def _get_inv(tensor):
    """Invert a tensor of shape (N, N, ...) pointwise."""
    T  = xp.moveaxis(tensor, [0, 1], [-2, -1])
    Ti = xp.linalg.inv(T)
    return xp.moveaxis(Ti, [-2, -1], [0, 1])

def get_c4_inv(g): return _get_inv(g)
def get_c3_inv(g): return _get_inv(g)


# ── Christoffel symbols ───────────────────────────────────────────────────────

def get_christoffel(metric_tensor, grid_scale):
    """
    Christoffel symbols  Γ^λ_{μν}  shape (4, 4, 4, Nt, Nx, Ny, Nz).

    Standard formula:
        Γ^λ_{μν} = ½ g^{λρ} (∂_μ g_{νρ} + ∂_ν g_{μρ} − ∂_ρ g_{μν})
    """
    g_inv  = get_c4_inv(metric_tensor)
    dt, dx, dy, dz = grid_scale
    deltas = [dt, dx, dy, dz]

    # d_g[rho, mu, nu] = ∂_rho g_{μν}
    d_g = xp.stack(
        [derive_1st_4th_order(metric_tensor, rho + 2, deltas[rho]) for rho in range(4)],
        axis=0
    )  # (4, 4, 4, ...)

    shape  = metric_tensor.shape
    gamma  = xp.zeros((4, 4, 4) + shape[2:])

    for lam in range(4):
        for mu in range(4):
            for nu in range(4):
                val = xp.zeros(shape[2:])
                for rho in range(4):
                    val += 0.5 * g_inv[lam, rho] * (
                        d_g[mu, nu, rho]
                        + d_g[nu, mu, rho]
                        - d_g[rho, mu, nu]
                    )
                gamma[lam, mu, nu] = val
    return gamma


# ── Ricci tensor ──────────────────────────────────────────────────────────────

def get_ricci_tensor(metric_tensor, grid_scale):
    """
    Ricci tensor  R_{μν}  shape (4, 4, Nt, Nx, Ny, Nz).

    R_{μν} = ∂_ρ Γ^ρ_{μν} − ∂_ν Γ^ρ_{μρ}
           + Γ^ρ_{μν} Γ^σ_{ρσ} − Γ^σ_{μρ} Γ^ρ_{νσ}
    """
    gamma  = get_christoffel(metric_tensor, grid_scale)
    dt, dx, dy, dz = grid_scale
    deltas = [dt, dx, dy, dz]
    shape  = metric_tensor.shape

    # Term 1: ∑_ρ ∂_ρ Γ^ρ_{μν}
    term1 = xp.zeros(shape)
    for rho in range(4):
        g_slice = gamma[rho, :, :]              # (4, 4, ...)
        term1  += derive_1st_4th_order(g_slice, rho + 2, deltas[rho])

    # Contracted Christoffel  V_μ = ∑_ρ Γ^ρ_{μρ}
    V = xp.zeros((4,) + shape[2:])
    for rho in range(4):
        V += gamma[rho, :, rho]

    # Term 2: ∂_ν V_μ
    term2 = xp.zeros(shape)
    for nu in range(4):
        d_nu_V = derive_1st_4th_order(V, nu + 1, deltas[nu])  # (4, ...)
        for mu in range(4):
            term2[mu, nu] = d_nu_V[mu]

    # Term 3: Γ^ρ_{μν} V_ρ
    term3 = xp.zeros(shape)
    for rho in range(4):
        term3 += gamma[rho, :, :] * V[rho]

    # Term 4 (BUG-FIXED): Γ^σ_{μρ} Γ^ρ_{νσ}  — outer product, not element-wise
    # For fixed σ, ρ:
    #   G1[μ] = Γ^σ_{μρ}  →  (4, ...)
    #   G2[ν] = Γ^ρ_{νσ}  →  (4, ...)
    #   contribution = G1[μ] * G2[ν]  →  (4, 4, ...)  via outer product
    term4 = xp.zeros(shape)
    for sigma in range(4):
        for rho in range(4):
            G1 = gamma[sigma, :, rho]                             # (4, ...)
            G2 = gamma[rho,   :, sigma]                           # (4, ...)
            term4 += G1[:, xp.newaxis, ...] * G2[xp.newaxis, :, ...]  # (4,4,...)

    return term1 - term2 + term3 - term4


# ── Ricci scalar ──────────────────────────────────────────────────────────────

def get_ricci_scalar(R_mn, g_inv):
    """Ricci scalar  R = g^{μν} R_{μν}."""
    R = xp.zeros(R_mn.shape[2:])
    for mu in range(4):
        for nu in range(4):
            R += R_mn[mu, nu] * g_inv[mu, nu]
    return R


# ── Einstein tensor ───────────────────────────────────────────────────────────

def get_einstein_tensor(R_mn, R, g):
    """Einstein tensor  G_{μν} = R_{μν} − ½ R g_{μν}."""
    G = xp.zeros(g.shape)
    for mu in range(4):
        for nu in range(4):
            G[mu, nu] = R_mn[mu, nu] - 0.5 * R * g[mu, nu]
    return G


# ── Stress-energy tensor ──────────────────────────────────────────────────────

def get_energy_tensor(G_mn, g_inv):
    """
    Stress-energy tensor  T^{μν}  (contravariant) from the Einstein tensor.

    G_{μν} = (8π G / C⁴) T_{μν}
    T^{μν} = (C⁴ / 8π G) g^{μα} g^{νβ} G_{αβ}
    """
    factor   = (C**4) / (8.0 * xp.pi * G)
    T_cov    = factor * G_mn

    # Raise both indices: T^{μν} = g^{μα} T_{αβ} g^{νβ}
    Tc = xp.moveaxis(T_cov,  [0, 1], [-2, -1])  # (..., 4, 4)
    Gi = xp.moveaxis(g_inv,  [0, 1], [-2, -1])  # (..., 4, 4)
    Tr = xp.matmul(xp.matmul(Gi, Tc), Gi)       # (..., 4, 4)
    return xp.moveaxis(Tr, [-2, -1], [0, 1])


def solve_energy_tensor(metric):
    """
    Full pipeline: metric → stress-energy tensor T^{μν}.
    Returns a new Metric-like object with type='Stress-Energy'.
    """
    g_inv = get_c4_inv(metric.tensor)
    R_mn  = get_ricci_tensor(metric.tensor, metric.scaling)
    R     = get_ricci_scalar(R_mn, g_inv)
    G_mn  = get_einstein_tensor(R_mn, R, metric.tensor)
    T_uv  = get_energy_tensor(G_mn, g_inv)

    m = Metric(
        tensor=T_uv,
        coords=metric.coords,
        scaling=metric.scaling,
        name=f'Energy Tensor of {metric.name}',
        index='contravariant',
        params=metric.params,
    )
    m.type = 'Stress-Energy'
    return m


print('Tensor calculation routines loaded.')

---
## Cell 7 — Frame transformations & energy conditions

In [ ]:
# ── Index raising / lowering ──────────────────────────────────────────────────

def change_tensor_index(tensor, new_index, metric):
    """Raise or lower both indices of a rank-2 tensor object."""
    if tensor.index == new_index:
        return tensor

    g = metric.tensor
    if metric.index == 'covariant':
        g_lower = g
        g_raise = get_c4_inv(g)
    else:
        g_raise = g
        g_lower = get_c4_inv(g)

    t_val = tensor.tensor
    Tmat  = xp.moveaxis(t_val, [0, 1], [-2, -1])

    if tensor.index == 'covariant' and new_index == 'contravariant':
        Gmat = xp.moveaxis(g_raise, [0, 1], [-2, -1])
        Rmat = xp.matmul(xp.matmul(Gmat, Tmat), Gmat)
    else:
        Gmat = xp.moveaxis(g_lower, [0, 1], [-2, -1])
        Rmat = xp.matmul(xp.matmul(Gmat, Tmat), Gmat)

    result = xp.moveaxis(Rmat, [-2, -1], [0, 1])
    new_t  = Metric(
        tensor=result,
        coords=tensor.coords,
        scaling=tensor.scaling,
        name=tensor.name,
        index=new_index,
        params=tensor.params,
    )
    if hasattr(tensor, 'type'):  new_t.type  = tensor.type
    if hasattr(tensor, 'frame'): new_t.frame = tensor.frame
    return new_t


# ── Eulerian transformation matrix (Cholesky decomposition) ──────────────────

def get_eulerian_transformation_matrix(g):
    """
    Compute the tetrad / vierbein matrix M such that M^T g M = η (Minkowski).
    Uses a recursive Cholesky-like factorisation ported from WarpFactory MATLAB.

    g has shape (4, 4, ...).  Returns M of the same shape.
    """
    shape = g.shape
    M = xp.zeros(shape)

    # MATLAB uses 1-based indices 1..4; Python uses 0..3.
    # Mapping:  MATLAB g(i,j)  →  Python g[i-1, j-1]
    g11,g12,g13,g14 = g[0,0], g[0,1], g[0,2], g[0,3]
    g22,g23,g24     = g[1,1], g[1,2], g[1,3]
    g33,g34         = g[2,2], g[2,3]
    g44             = g[3,3]

    factor0 = g44
    factor1 = -g34**2 + g33 * factor0
    factor2 = (2*g23*g24*g34 - g44*g23**2 - g33*g24**2 + g22*factor1)

    t_huge = (
        -2*g44*g12*g13*g23 + 2*g13*g14*g23*g24 + 2*g12*g13*g24*g34
        + 2*g12*g14*g23*g34 - g12**2*g34**2 - g13**2*g24**2 - g14**2*g23**2
        + g33*(-2*g12*g14*g24 + g44*g12**2)
        + g22*(-2*g13*g14*g34 + g44*g13**2 + g33*g14**2)
    )
    factor3 = t_huge - g11 * factor2

    eps = 1e-30  # guard against division by zero

    M[3, 3] = xp.sqrt(xp.abs(1.0 / (factor0 + eps))) * xp.sign(factor0 + eps)
    M[2, 2] = xp.sqrt(xp.abs(factor0 / (factor1 + eps)))
    M[3, 2] = -g34 / xp.sqrt(xp.abs(factor0 * factor1) + eps)

    M[1, 1] = xp.sqrt(xp.abs(factor1 / (factor2 + eps)))
    M[2, 1] = (g24*g34 - g23*g44) / xp.sqrt(xp.abs(factor1 * factor2) + eps)
    M[3, 1] = (g23*g34 - g24*g33) / xp.sqrt(xp.abs(factor1 * factor2) + eps)

    M[0, 0] = xp.sqrt(xp.abs(factor2 / (factor3 + eps)))

    n21 = (g12*g34**2 + g13*g23*g44 - g13*g24*g34
           - g14*g23*g34 + g14*g24*g33 - g12*g33*g44)
    M[1, 0] = n21 / xp.sqrt(xp.abs(factor2 * factor3) + eps)

    n31 = (g13*g24**2 - g14*g23*g24 + g12*g23*g44
           - g12*g24*g34 - g13*g22*g44 + g14*g22*g34)
    M[2, 0] = n31 / xp.sqrt(xp.abs(factor2 * factor3) + eps)

    n41 = (g14*g23**2 - g13*g23*g24 - g12*g23*g34
           + g12*g24*g33 + g13*g22*g34 - g14*g22*g33)
    M[3, 0] = n41 / xp.sqrt(xp.abs(factor2 * factor3) + eps)

    return M


# ── Eulerian frame transfer ───────────────────────────────────────────────────

def do_frame_transfer(metric, energy_tensor, frame='Eulerian'):
    """
    Project the energy tensor into the locally-flat Eulerian reference frame.

    Transformation:  T̂_{ab} = M^T_{a}^{μ}  T_{μν}  M^{ν}_{b}
    Then raise indices using the local Minkowski metric.
    """
    if frame != 'Eulerian':
        raise ValueError('Only Eulerian frame is supported.')
    if hasattr(energy_tensor, 'frame') and energy_tensor.frame == 'Eulerian':
        return energy_tensor

    # 1. Ensure covariant T_{μν}
    if energy_tensor.index != 'covariant':
        T_cov = change_tensor_index(energy_tensor, 'covariant', metric)
    else:
        T_cov = energy_tensor

    # 2. Tetrad matrix M
    M = get_eulerian_transformation_matrix(metric.tensor)

    Mm  = xp.moveaxis(M, [0, 1], [-2, -1])          # (..., 4, 4)
    MmT = xp.swapaxes(Mm, -1, -2)                    # (..., 4, 4)  (transpose)
    Tm  = xp.moveaxis(T_cov.tensor, [0, 1], [-2, -1])

    Res = xp.matmul(xp.matmul(MmT, Tm), Mm)
    result_val = xp.moveaxis(Res, [-2, -1], [0, 1])

    # 3. Raise time-space cross indices with Minkowski sign convention
    rc = result_val.copy()
    for i in range(1, 4):
        rc[0, i] = -rc[0, i]
        rc[i, 0] = -rc[i, 0]

    new_t = Metric(
        tensor=rc,
        coords=energy_tensor.coords,
        scaling=energy_tensor.scaling,
        name=f'{energy_tensor.name} (Eulerian)',
        index='contravariant',
        params=energy_tensor.params,
    )
    new_t.frame = 'Eulerian'
    new_t.type  = 'Stress-Energy'
    return new_t


# ── Energy conditions ─────────────────────────────────────────────────────────

def _sample_null_vectors(num_vecs):
    """Fibonacci-lattice null vectors on the unit sphere (numpy, for CPU portability)."""
    idx   = np.arange(num_vecs, dtype=float) + 0.5
    phi   = np.arccos(1 - 2 * idx / num_vecs)
    theta = np.pi * (1 + 5**0.5) * idx
    vecs  = np.zeros((4, num_vecs))
    vecs[0] = 1.0
    vecs[1] = np.cos(theta) * np.sin(phi)
    vecs[2] = np.sin(theta) * np.sin(phi)
    vecs[3] = np.cos(phi)
    return vecs


def _lower_minkowski(T_upper):
    """Lower both indices with η = diag(-1,+1,+1,+1)."""
    T = T_upper.copy()
    T[0, :] = -T_upper[0, :]
    T[:, 0] = -T_upper[:, 0]
    T[0, 0] =  T_upper[0, 0]
    return T


def calculate_energy_conditions(metric_tensor, grid_scale, num_vecs=50):
    """
    Full pipeline: metric → energy conditions.

    Parameters
    ----------
    metric_tensor : array (4, 4, Nt, Nx, Ny, Nz) — covariant metric g_{μν}.
    grid_scale    : (dt, dx, dy, dz).
    num_vecs      : null vectors sampled on the sphere (default 50).

    Returns
    -------
    results  : dict with keys 'Null' and 'Weak'.
               Shape (Nt, Nx, Ny, Nz).  Values < 0 indicate a violation.
    T_euler  : contravariant stress-energy tensor in the Eulerian frame,
               shape (4, 4, Nt, Nx, Ny, Nz).  T_euler[0,0] = energy density ρ.
    """
    # Step 1: solve Einstein equations
    g_inv = get_c4_inv(metric_tensor)
    R_mn  = get_ricci_tensor(metric_tensor, grid_scale)
    R_sc  = get_ricci_scalar(R_mn, g_inv)
    G_mn  = get_einstein_tensor(R_mn, R_sc, metric_tensor)
    T_uv  = get_energy_tensor(G_mn, g_inv)

    # Step 2: wrap in Metric objects
    sp    = metric_tensor.shape[2:]
    _z    = xp.zeros(sp)
    dc    = {'t': _z, 'x': _z, 'y': _z, 'z': _z}
    sc    = xp.array(grid_scale, dtype=float)

    met   = Metric(tensor=metric_tensor, coords=dc, scaling=sc,
                   name='metric', index='covariant')
    eng   = Metric(tensor=T_uv, coords=dc, scaling=sc,
                   name='energy_tensor', index='contravariant')
    eng.type = 'Stress-Energy'

    # Step 3: Eulerian frame transfer
    eng_eu  = do_frame_transfer(met, eng, 'Eulerian')
    T_euler = eng_eu.tensor

    # Step 4: lower indices with η
    T_low = _lower_minkowski(T_euler)

    # Step 5: NEC — min over null directions
    vecs_np = _sample_null_vectors(num_vecs)          # always numpy
    T_end   = xp.moveaxis(T_low, [0, 1], [-2, -1])   # (..., 4, 4)
    nec_map = xp.full(T_low.shape[2:], xp.inf)

    for i in range(num_vecs):
        k   = xp.asarray(vecs_np[:, i])               # (4,)
        val = (T_end @ k) @ k                          # (...)
        nec_map = xp.minimum(nec_map, val)

    # Step 6: WEC = min(ρ, NEC)
    rho     = T_euler[0, 0]
    wec_map = xp.minimum(rho, nec_map)

    return {'Null': nec_map, 'Weak': wec_map}, T_euler


print('Frame transformations and energy conditions loaded.')

---
## Demo 1 — Alcubierre Warp Drive

The **Alcubierre metric** (1994) describes a hypothetical warp bubble that contracts  
spacetime in front and expands it behind a spacecraft, allowing faster-than-light travel  
without local violation of special relativity.

**Expected result**: The warp shell requires *negative energy density* (exotic matter),  
which shows up as a violation of both the Null and Weak energy conditions.

In [ ]:
import time

# ── Grid parameters ────────────────────────────────────────────────────────────
# Nt >= 5 is required for the 4th-order finite-difference stencil.
# Increase grid size for higher resolution (uses more memory / time).
# Recommended: start with 25×25×25 (fast), go up to 50×50×50 on GPU.

Nt, Nx, Ny, Nz = 7, 25, 25, 25       # grid dimensions
dt, dx, dy, dz  = 1.0, 0.3, 0.3, 0.3  # grid spacing  (code units)
grid_size        = (Nt, Nx, Ny, Nz)
grid_scale       = (dt, dx, dy, dz)

# World centre — put bubble at the middle of the spatial grid
xc = dx * Nx / 2
yc = dy * Ny / 2
zc = dz * Nz / 2
world_center = (dt, xc, yc, zc)

# Warp drive physics
v     = 0.5   # bubble speed  (0.5 = half light-speed in code units)
R     = 1.5   # bubble radius (code units)
sigma = 8.0   # shell steepness (larger = thinner shell)

# ── Generate metric ────────────────────────────────────────────────────────────
print('Creating Alcubierre metric ...')
t0 = time.time()
metric_alc = create_alcubierre_metric(grid_size, grid_scale, world_center, v, R, sigma)
print(f'  Done in {time.time()-t0:.2f} s   tensor shape: {metric_alc.tensor.shape}')

# ── Solve Einstein equations + energy conditions ──────────────────────────────
print('Solving Einstein equations and computing energy conditions ...')
print('  (This is the heavy step — GPU helps for larger grids.)')
t0 = time.time()
results_alc, T_euler_alc = calculate_energy_conditions(
    metric_alc.tensor, grid_scale
)
print(f'  Done in {time.time()-t0:.2f} s')

# ── Summary statistics ─────────────────────────────────────────────────────────
t_sl, z_sl = Nt // 2, Nz // 2

rho    = to_np(T_euler_alc[0, 0, t_sl, :, :, z_sl])
nec_2d = to_np(results_alc['Null'][t_sl, :, :, z_sl])
wec_2d = to_np(results_alc['Weak'][t_sl, :, :, z_sl])

print(f'\nEnergy density  — max: {rho.max():.4e}   min: {rho.min():.4e}')
print(f'NEC min value   : {nec_2d.min():.4e}  (negative = violated)')
print(f'WEC min value   : {wec_2d.min():.4e}  (negative = violated)')

if rho.min() < 0:
    print('\nSUCCESS  ✓  Negative energy density detected — expected for Alcubierre warp drive.')
else:
    print('\nWARNING  ✗  No negative energy detected — check grid resolution or parameters.')

In [ ]:
# ── Visualisation ──────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
ext  = [0, Nx * dx, 0, Ny * dy]
kwargs = dict(origin='lower', extent=ext, cmap='RdBu_r')

im0 = axes[0].imshow(rho.T, **kwargs)
plt.colorbar(im0, ax=axes[0])
axes[0].set_title('Energy Density  ρ = T̂⁰⁰\n(negative = exotic matter required)',
                  fontsize=10)
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# Use a diverging colormap centred on 0 for violation maps
lim = max(abs(nec_2d.min()), abs(nec_2d.max()), 1e-30)
im1 = axes[1].imshow(nec_2d.T, vmin=-lim, vmax=lim, **kwargs)
plt.colorbar(im1, ax=axes[1])
axes[1].set_title('Null Energy Condition  (NEC)\nmin T_{μν}k^μk^ν  —  blue < 0 = violated',
                  fontsize=10)
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')

lim = max(abs(wec_2d.min()), abs(wec_2d.max()), 1e-30)
im2 = axes[2].imshow(wec_2d.T, vmin=-lim, vmax=lim, **kwargs)
plt.colorbar(im2, ax=axes[2])
axes[2].set_title('Weak Energy Condition  (WEC)\nmin(ρ, NEC)  —  blue < 0 = violated',
                  fontsize=10)
axes[2].set_xlabel('x'); axes[2].set_ylabel('y')

fig.suptitle(
    f'Alcubierre Warp Drive  |  v={v}c  R={R}  σ={sigma}  '
    f'grid {Nx}³  (t-slice {t_sl}, z-slice {z_sl})',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig('alcubierre_energy_conditions.png', dpi=130)
plt.show()
print('Figure saved to alcubierre_energy_conditions.png')

In [ ]:
# ── Shape function visualisation ───────────────────────────────────────────────

r_vals = np.linspace(0, 3 * R, 300)
f_vals = to_np(shape_function_alcubierre(
    xp.asarray(r_vals), R, sigma
))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r_vals, f_vals, linewidth=2.5)
ax.axvline(R, color='red', linestyle='--', alpha=0.7, label=f'R = {R}')
ax.axhline(0.5, color='grey', linestyle=':', alpha=0.5)
ax.set_xlabel('r  (code units)')
ax.set_ylabel('f(r)')
ax.set_title(f'Alcubierre shape function  (R={R}, σ={sigma})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('alcubierre_shape_function.png', dpi=130)
plt.show()

---
## Demo 2 — Schwarzschild Black Hole

The **Schwarzschild metric** is the unique spherically symmetric vacuum solution.  
It has zero stress-energy everywhere outside the horizon (vacuum solution),  
so energy conditions are trivially satisfied.

In [ ]:
# Schwarzschild radius: rs = 2 G M / C²
# In code units (G=C=1): rs = 2 M, so M = rs/2.
rs     = 1.0       # Schwarzschild radius in code units

# Use a single time slice (static metric)
Nt_s, Nx_s, Ny_s, Nz_s = 1, 25, 25, 25
dx_s = 0.3
grid_size_s  = (Nt_s, Nx_s, Ny_s, Nz_s)
grid_scale_s = (1.0, dx_s, dx_s, dx_s)

# Centre the grid; the singularity will be at the spatial origin
xc_s  = dx_s * Nx_s / 2
world_center_s = (1.0, xc_s, xc_s, xc_s)

print('Creating Schwarzschild metric ...')
metric_sch = create_schwarzschild_metric(
    grid_size_s, grid_scale_s, world_center_s, rs
)
print(f'  tensor shape: {metric_sch.tensor.shape}')

# Visualise g_tt component
z_sl_s = Nz_s // 2
g_tt   = to_np(metric_sch.tensor[0, 0, 0, :, :, z_sl_s])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ext_s = [0, Nx_s * dx_s, 0, Ny_s * dx_s]
im0 = axes[0].imshow(g_tt.T, origin='lower', extent=ext_s, cmap='plasma')
plt.colorbar(im0, ax=axes[0])
axes[0].set_title(f'g_{{tt}} = -(1 - r_s/r)  |  r_s = {rs}', fontsize=10)
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# Radial profile
mid  = Ny_s // 2
r_pr = np.linspace(1, Nx_s, Nx_s) * dx_s - xc_s
g_pr = g_tt[:, mid]

axes[1].plot(to_np(metric_sch.coords['x'][:, mid, z_sl_s]), g_pr, linewidth=2)
axes[1].axhline(-1, color='grey', linestyle=':', label='Minkowski g_tt = -1')
axes[1].axvline(0,  color='red',  linestyle='--', label='singularity')
axes[1].set_xlabel('x (code units)'); axes[1].set_ylabel('g_tt')
axes[1].set_title('Radial profile of g_tt')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.suptitle(f'Schwarzschild Metric  |  r_s = {rs}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('schwarzschild_metric.png', dpi=130)
plt.show()
print('Figure saved to schwarzschild_metric.png')

---
## Demo 3 — Van Den Broeck Metric

The **Van Den Broeck (1999)** metric modifies the Alcubierre drive by  
expanding the interior of the bubble.  This dramatically reduces the total  
exotic energy required compared to the standard Alcubierre design.

In [ ]:
print('Creating Van Den Broeck metric ...')

Nt_v, Nx_v, Ny_v, Nz_v = 7, 25, 25, 25
dx_v = 0.3
grid_size_v  = (Nt_v, Nx_v, Ny_v, Nz_v)
grid_scale_v = (1.0, dx_v, dx_v, dx_v)

xc_v = dx_v * Nx_v / 2
world_center_v = (1.0, xc_v, xc_v, xc_v)

# Van Den Broeck parameters
v_vdb    = 0.5    # velocity (units of C)
R1_vdb   = 2.5    # spatial expansion radius
sigma1   = 6.0    # expansion steepness
R2_vdb   = 1.0    # shift vector radius (inner warp shell)
sigma2   = 8.0    # shift steepness
A_vdb    = 2.0    # spatial expansion factor

metric_vdb = create_van_den_broeck_metric(
    grid_size_v, grid_scale_v, world_center_v,
    v_vdb, R1_vdb, sigma1, R2_vdb, sigma2, A_vdb
)
print(f'  tensor shape: {metric_vdb.tensor.shape}')

# Visualise g_tt component
z_sl_v = Nz_v // 2
t_sl_v = Nt_v // 2
g_tt_v = to_np(metric_vdb.tensor[0, 0, t_sl_v, :, :, z_sl_v])
g_xx_v = to_np(metric_vdb.tensor[1, 1, t_sl_v, :, :, z_sl_v])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ext_v = [0, Nx_v * dx_v, 0, Ny_v * dx_v]

im0 = axes[0].imshow(g_tt_v.T, origin='lower', extent=ext_v, cmap='RdBu_r')
plt.colorbar(im0, ax=axes[0])
axes[0].set_title('g_tt  (Van Den Broeck)', fontsize=10)
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

im1 = axes[1].imshow(g_xx_v.T, origin='lower', extent=ext_v, cmap='viridis')
plt.colorbar(im1, ax=axes[1])
axes[1].set_title('g_xx = B²  (spatial expansion)', fontsize=10)
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')

fig.suptitle(
    f'Van Den Broeck Metric  |  v={v_vdb}c  A={A_vdb}  '
    f'R1={R1_vdb}  R2={R2_vdb}',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig('van_den_broeck_metric.png', dpi=130)
plt.show()
print('Figure saved to van_den_broeck_metric.png')

---
## GPU performance benchmark (optional)

In [ ]:
# Run this cell to compare CPU vs GPU timing on a larger grid.
# Requires CuPy to be installed (GPU runtime in Colab).

import time

def run_alcubierre_benchmark(grid_n, use_gpu=True):
    """Time the full Alcubierre pipeline on a grid_n^3 spatial grid."""
    global xp, GPU_AVAILABLE
    orig_xp = xp

    if use_gpu and GPU_AVAILABLE:
        import cupy as _cp
        xp = _cp
        label = 'GPU (CuPy)'
    else:
        import numpy as _np
        xp = _np
        label = 'CPU (NumPy)'

    gs = (7, grid_n, grid_n, grid_n)
    sc = (1.0, 0.3, 0.3, 0.3)
    wc = (1.0, 0.3*grid_n/2, 0.3*grid_n/2, 0.3*grid_n/2)

    t0 = time.time()
    m  = create_alcubierre_metric(gs, sc, wc, 0.5, 1.5, 8.0)
    _, _ = calculate_energy_conditions(m.tensor, sc)
    elapsed = time.time() - t0

    xp = orig_xp  # restore
    print(f'  {label:20s}  grid {grid_n}³   time: {elapsed:.2f} s')
    return elapsed


print('Benchmark: full Alcubierre pipeline')
print('-' * 45)
t_cpu = run_alcubierre_benchmark(20, use_gpu=False)

if GPU_AVAILABLE:
    t_gpu = run_alcubierre_benchmark(20, use_gpu=True)
    print(f'\n  Speedup: {t_cpu/t_gpu:.1f}×  (GPU / CPU)')
else:
    print('\n  GPU not available — no comparison.')

---
## Tips & further reading

### Parameter guide

| Parameter | Effect | Typical range |
|-----------|--------|---------------|
| `v`  | Bubble speed (units of C) | 0.1 – 2.0 |
| `R`  | Bubble radius (code units) | 1 – 5 |
| `sigma` | Shell steepness | 4 – 20 |
| `Nx,Ny,Nz` | Grid resolution | ≥ 25 recommended |
| `Nt` | Time slices | ≥ 5 (required for finite diff.) |

### Grid size vs GPU benefit

| Grid size | CPU time (est.) | GPU benefit |
|-----------|-----------------|-------------|
| 20³ | ~ 10 s | minimal (overhead dominates) |
| 40³ | ~ 1 min | moderate (2–5×) |
| 60³ | ~ 5 min | significant (5–15×) |

### Understanding the output

- **Energy density `ρ = T̂⁰⁰`**: the energy seen by a local (Eulerian) observer.  
  Negative ρ means *exotic matter* is required — a key feature of warp drive spacetimes.
- **NEC (Null Energy Condition)**: `T_{μν} k^μ k^ν ≥ 0` for all null k.  
  Alcubierre and Van Den Broeck *both violate* this.
- **WEC (Weak Energy Condition)**: energy density ≥ 0 for all timelike observers.  
  A stricter condition; also violated in warp drive metrics.

### Using your own metric

```python
# Any metric with shape (4, 4, Nt, Nx, Ny, Nz) works:
my_g = xp.zeros((4, 4, Nt, Nx, Ny, Nz))
# fill in your metric components …
results, T_euler = calculate_energy_conditions(my_g, grid_scale)
```

### References

- Alcubierre (1994): *The warp drive: hyper-fast travel within general relativity*
- Van Den Broeck (1999): *A warp drive with more reasonable total energy requirements*
- WarpFactory MATLAB toolkit: original codebase this notebook ports